In [16]:
import numpy as np
import matplotlib.pyplot as plt
import random
import isotp # thanks to Liz for figuring this out

In [2]:
from parsing_lib import frame_and_muid, populate_dict, populate_dict_panda, calculate_unique_message_id
from parsing_lib import return_all_messages_for_frame, return_all_frames_messages_for_muid_and_frame
from parsing_lib import return_frame_IDs_with_message_changes, return_frame_IDs_muids, return_unique_messages_for_frame
from parsing_lib import return_frame_IDs_muids_with_message_changes_in_timestamp_range, return_interesting_timestamps

In [3]:
%ls ../CAN_logs/panda

I-CAN_car_in_D.csv
I-CAN_car_in_ready_parked.csv
I-CAN_nothing_on.csv
maybe_too_short/
M-CAN_nav_start_to_bank_0.5mi.csv
M-CAN_nav_start_to_chicken_5.3mi.csv
M-CAN_nav_start_to_Fresno_939mi.csv
M-CAN_nav_start_to_school_1.6mi_2.csv
M-CAN_panda_nothing_on_climate_start_both_warmers.csv
M-CAN_panda_nothing_on_climate_start_no_warmers.csv
M-CAN_panda_nothing_on_climate_stop.csv
M-CAN_panda_nothing_on.csv
M-CAN_panda_nothing_on_remote_lock.csv
M-CAN_panda_nothing_on_remote_unlock.csv


In [4]:
d420mi = {}
d1p1mi = {}
d1p6mi = {}
d2p1mi = {}
s1p6mi = {}
s939mi = {}
s0p5mi = {}
s5p3mi = {}

In [5]:
populate_dict_panda(d420mi, '../CAN_logs/panda/maybe_too_short/M-CAN_nav_already_on_to_addr_420mi.csv')
populate_dict_panda(d1p1mi, '../CAN_logs/panda/maybe_too_short/M-CAN_nav_already_on_to_addr_1.1mi.csv')
populate_dict_panda(d1p6mi, '../CAN_logs/panda/maybe_too_short/M-CAN_nav_already_on_to_school_1.6mi.csv')
populate_dict_panda(d2p1mi, '../CAN_logs/panda/maybe_too_short/M-CAN_nav_already_on_to_addr_2.1mi.csv')
populate_dict_panda(s1p6mi, '../CAN_logs/panda/M-CAN_nav_start_to_school_1.6mi_2.csv')
populate_dict_panda(s0p5mi, '../CAN_logs/panda/M-CAN_nav_start_to_bank_0.5mi.csv')
populate_dict_panda(s5p3mi, '../CAN_logs/panda/M-CAN_nav_start_to_chicken_5.3mi.csv')
populate_dict_panda(s939mi, '../CAN_logs/panda/M-CAN_nav_start_to_Fresno_939mi.csv')

In [6]:
all_logs = [
d420mi,
d1p1mi,
d1p6mi,
d2p1mi,
s1p6mi,
s0p5mi,
s5p3mi,
s939mi
]

In [7]:
for log in all_logs:
    calculate_unique_message_id(log)

In [8]:
d420_frames_muids = return_frame_IDs_muids(d420mi, bus=2)
d1p1_frames_muids = return_frame_IDs_muids(d1p1mi, bus=2)
d1p6_frames_muids = return_frame_IDs_muids(d1p6mi, bus=2)
d2p1_frames_muids = return_frame_IDs_muids(d2p1mi, bus=2)
s1p6_frames_muids = return_frame_IDs_muids(s1p6mi, bus=2)
s0p5_frames_muids = return_frame_IDs_muids(s0p5mi, bus=2)
s5p3_frames_muids = return_frame_IDs_muids(s5p3mi, bus=2)
s939_frames_muids = return_frame_IDs_muids(s939mi, bus=2)

In [9]:
start_diff = np.setxor1d(s1p6_frames_muids, d1p6_frames_muids)

In [10]:
diff = s939_frames_muids
for frames_muids in [s1p6_frames_muids, s0p5_frames_muids, s5p3_frames_muids]:
    diff = np.setxor1d(diff, frames_muids)

In [11]:
s0p5mi["frames_muids"] = s0p5_frames_muids
s1p6mi["frames_muids"] = s1p6_frames_muids
s5p3mi["frames_muids"] = s5p3_frames_muids
s939mi["frames_muids"] = s939_frames_muids

In [12]:
for log in [s0p5mi, s1p6mi, s5p3mi, s939mi]:
    print(log["filename"])
    for frame_muid in diff:
        if(frame_muid in log["frames_muids"] and frame_muid.frame_id == 1767):
            # print(frame_muid)
            print(f"Frame ID: {frame_muid.frame_id}; Message: {log["messages"][log["messages_unique_ids"] == frame_muid.muid][0]}")

../CAN_logs/panda/M-CAN_nav_start_to_bank_0.5mi.csv
Frame ID: 1767; Message: [ 16  18 244   0  75   0 101   0]
Frame ID: 1767; Message: [ 33 121   0  32   0  66   0  97]
Frame ID: 1767; Message: [ 34   0 110   0 107   0 170 170]
../CAN_logs/panda/M-CAN_nav_start_to_school_1.6mi_2.csv
Frame ID: 1767; Message: [ 16  76 244   0  72   0 101   0]
Frame ID: 1767; Message: [ 33 110   0 114   0 121   0  32]
Frame ID: 1767; Message: [ 34   0  68   0  97   0 118   0]
Frame ID: 1767; Message: [ 35 105   0 100   0  32   0  84]
Frame ID: 1767; Message: [ 36   0 104   0 111   0 114   0]
Frame ID: 1767; Message: [ 37 101   0  97   0 117   0  32]
Frame ID: 1767; Message: [ 38   0  69   0 108   0 101   0]
Frame ID: 1767; Message: [ 39 109   0 101   0 110   0 116]
Frame ID: 1767; Message: [ 40   0  97   0 114   0 121   0]
Frame ID: 1767; Message: [ 41  32   0  83   0  99   0 104]
Frame ID: 1767; Message: [ 42   0 111   0 111   0 108   0]
../CAN_logs/panda/M-CAN_nav_start_to_chicken_5.3mi.csv
Frame ID: 1

In [100]:
unique_ids = np.unique([frame_muid.frame_id for frame_muid in diff])

In [106]:
# for frame_id in unique_ids:
for frame_id in [1767]:
    print(frame_id)
    for log in [s1p6mi, s5p3mi, s0p5mi, s939mi]:
        print(log['filename'])
        for message in return_unique_messages_for_frame(log,frame_id):
            print(message)

1767
../CAN_logs/panda/M-CAN_nav_start_to_school_1.6mi_2.csv
[  3 242   0   0 170 170 170 170]
[ 16  21 240  55   0  50   0 110]
[ 16  29 242   3  67  67  67  55]
[ 16  76 244   0  72   0 101   0]
[ 33   0  50   0 110   0 100   0]
[ 33   0 100   0  32   0  80   0]
[ 33 110   0 114   0 121   0  32]
[ 34   0  68   0  97   0 118   0]
[ 34  32   0  80   0 108   0  32]
[ 34 108   0  32   0  78   0  69]
[35  0 78  0 69  0  0  0]
[ 35   0 170 170 170 170 170 170]
[ 35 105   0 100   0  32   0  84]
[ 36   0   0 170 170 170 170 170]
[ 36   0 104   0 111   0 114   0]
[ 37 101   0  97   0 117   0  32]
[ 38   0  69   0 108   0 101   0]
[ 39 109   0 101   0 110   0 116]
[ 40   0  97   0 114   0 121   0]
[ 41  32   0  83   0  99   0 104]
[ 42   0 111   0 111   0 108   0]
../CAN_logs/panda/M-CAN_nav_start_to_chicken_5.3mi.csv
[  3 242   0   0 170 170 170 170]
[ 16  21 240  55   0  50   0 110]
[ 16  29 242   3  67  67  67  55]
[ 16  52 244   0  72   0 101   0]
[ 33   0  50   0 110   0 100   0]
[ 33   0

In [59]:
# this doesn't work yet
class isotp_reader(object):
    def __init__(self, log, frame_id):
        self.log = log
        self.frame_id = frame_id
        if(log.get('hex_message_block',None).any()):
            self.messages = log['hex_message_block'][log['ids'] == frame_id]
        else:
            self.messages_split = log['hex_messages'][log['ids'] == frame_id]
            self.messages = []
            for message in self.messages_split:
                new_message = b''
                for i in range(8):
                    new_message += message[i]
                self.messages.append(new_messsage)
        self.i = 0
    def read(self,timeout):
        if(self.i >= len(self.messages)):
            return None
        else:
            return_message = isotp.can_message.CanMessage(arbitration_id=self.frame_id,dlc=8,data=self.messages[self.i])
            self.i += 1
            return return_message 

In [104]:
s1p6_reader = isotp_reader(s1p6mi, 1767)
tl = isotp.TransportLayer(rxfn=s1p6_reader.read,txfn=None,address=isotp.Address(rxid=1767,txid=0),read_timeout=1)

In [105]:
tl.start()
recv=True
while(recv==True):
    recv = tl.recv()
    print(recv)
tl.stop()

Received a FlowControl message while transmission was Idle. Ignoring
Received a FlowControl message while transmission was Idle. Ignoring


None
